# 45. 2단계 판정 — 1단계 검사 대상(10%) → 2단계 그 안에서 불량/정상 판정

| 모델 | 1단계 (검사 대상 10%) | 2단계 (불량 판정) |
|---|---|---|
| CN7 경보 + std4 | 최종 모델 경보 샷 + 나머지를 std4 흔들림 순으로 | 최종 pct ≥ 0.999 **또는** std4 pct ≥ q |
| RG3 가스 전용 RF | 가스 RF 상위 10% | 가스 RF pct ≥ q |
| RG3 가스 RF + 미성형 N1 | 68%는 가스 RF 상위, 나머지는 미성형 N1 순위 | 가스 RF pct ≥ q **또는** 미성형 N1 pct ≥ q |

- **pct** = 정상 기준 데이터(unlabeled)에서 같은 모델 점수의 백분위. 지도 모델은 Run 하나씩 빼고 학습한 그 fold의 모델로 정상 데이터에도 점수를 매겨서 만듦 → "정상 생산 상위 1%(q=0.99)보다 이상하면 불량"
- q = 0.99 / 0.995 / 0.999 (미리 정함)
- **Precision(판정)**: 불량 판정한 샷 중 실제 불량 / **Recall(검사 대상 안)**: 1단계에 들어온 불량 중 판정으로 잡은 비율 / **Recall(전체 불량)**: 전체 불량 대비
- Accuracy는 정상이 대부분이라 의미가 없어 뺌

**준비**: 20a·20b 실행. RG3는 시드 3개 학습이라 수 분

In [1]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
import two_stage as TS
R = TS.run(seeds=(0, 1, 2))

## CN7 경보 + std4

In [2]:
display(R['cn7'].round(3))

,모델,q,검사 대상 샷,그 안의 불량,불량 판정 수,TP,FP,FN,TN,Precision(판정),Recall(검사 대상 안),Recall(전체 불량),F1(전체 기준),1단계 가스,판정 가스,1단계 미성형,판정 미성형
0,CN7 경보 + std4,0.990,198,8/15,53,3,50,5,140,0.057,0.375,0.2,0.088,5/11,0/11,3/4,3/4
1,CN7 경보 + std4,0.995,198,8/15,39,3,36,5,154,0.077,0.375,0.2,0.111,5/11,0/11,3/4,3/4
2,CN7 경보 + std4,0.999,198,8/15,15,3,12,5,178,0.200,0.375,0.2,0.200,5/11,0/11,3/4,3/4
3,CN7 경보 + std4 (2단계 = 최종 경보만),0.999,198,8/15,8,3,5,5,185,0.375,0.375,0.2,0.261,5/11,0/11,3/4,3/4


## RG3 가스 전용 RF / 가스 RF + 미성형 N1 (시드 평균)

In [3]:
display(R['rg3'].round(3))

,모델,q,검사 대상 샷,불량 판정 수,TP,FP,FN,TN,Precision(판정),Recall(검사 대상 안),Recall(전체 불량),F1(전체 기준),"그 안의 불량 (/25, 평균)","1단계 가스 (/17, 평균)","판정 가스 (/17, 평균)","1단계 미성형 (/8, 평균)","판정 미성형 (/8, 평균)"
0,RG3 가스 전용 RF,0.990,60.0,14.667,2.333,12.333,7.000,38.333,0.159,0.248,0.093,0.118,9.333,7.667,2.000,1.667,0.333
1,RG3 가스 RF + 미성형 N1,0.990,60.0,28.667,5.333,23.333,3.667,27.667,0.187,0.594,0.213,0.199,9.000,6.000,3.000,3.000,2.333
2,RG3 가스 전용 RF,0.995,60.0,7.333,1.000,6.333,8.333,44.333,0.139,0.107,0.040,0.062,9.333,7.667,1.000,1.667,0.000
3,RG3 가스 RF + 미성형 N1,0.995,60.0,9.000,1.333,7.667,7.667,43.333,0.148,0.145,0.053,0.077,9.000,6.000,1.000,3.000,0.333
4,RG3 가스 전용 RF,0.999,60.0,0.333,0.333,0.000,9.000,50.667,0.333,0.037,0.013,0.026,9.333,7.667,0.333,1.667,0.000
5,RG3 가스 RF + 미성형 N1,0.999,60.0,0.333,0.333,0.000,8.667,51.000,0.333,0.037,0.013,0.026,9.000,6.000,0.333,3.000,0.000


### 시드별 상세

In [4]:
display(R['rg3_by_seed'].round(3))

,모델,q,검사 대상 샷,그 안의 불량,불량 판정 수,TP,FP,FN,TN,Precision(판정),Recall(검사 대상 안),Recall(전체 불량),F1(전체 기준),1단계 가스,판정 가스,1단계 미성형,판정 미성형,seed
0,RG3 가스 전용 RF,0.990,60,10/25,15,3,12,7,38,0.200,0.300,0.12,0.150,8/17,2/17,2/8,1/8,0
1,RG3 가스 RF + 미성형 N1,0.990,60,8/25,31,5,26,3,26,0.161,0.625,0.20,0.179,5/17,3/17,3/8,2/8,0
2,RG3 가스 전용 RF,0.995,60,10/25,8,1,7,9,43,0.125,0.100,0.04,0.061,8/17,1/17,2/8,0/8,0
3,RG3 가스 RF + 미성형 N1,0.995,60,8/25,9,1,8,7,44,0.111,0.125,0.04,0.059,5/17,1/17,3/8,0/8,0
4,RG3 가스 전용 RF,0.999,60,10/25,0,0,0,10,50,0.000,0.000,0.00,0.000,8/17,0/17,2/8,0/8,0
5,RG3 가스 RF + 미성형 N1,0.999,60,8/25,0,0,0,8,52,0.000,0.000,0.00,0.000,5/17,0/17,3/8,0/8,0
6,RG3 가스 전용 RF,0.990,60,9/25,14,2,12,7,39,0.143,0.222,0.08,0.103,8/17,2/17,1/8,0/8,1
7,RG3 가스 RF + 미성형 N1,0.990,60,10/25,27,6,21,4,29,0.222,0.600,0.24,0.231,7/17,3/17,3/8,3/8,1
8,RG3 가스 전용 RF,0.995,60,9/25,8,1,7,8,44,0.125,0.111,0.04,0.061,8/17,1/17,1/8,0/8,1
9,RG3 가스 RF + 미성형 N1,0.995,60,10/25,12,2,10,8,40,0.167,0.200,0.08,0.108,7/17,1/17,3/8,1/8,1
